# ISLES'24 NCCT + perfusion ablation — preparation

CPU-only preparation for the second modernization experiment. This notebook extracts the reusable acute core subset from the verified `train.7z`, audits **NCCT + Tmax + CBF + CBV + MTT** on all 149 public training cases, stages `Dataset505_ISLES2024_NCCT_PERFUSION`, and refuses to continue unless its five patient folds are **exactly identical** to the frozen NCCT-only Dataset504 folds.

CTA and baseline clinical files are extracted at the same time for later controlled ablations, but they are **not inputs to Dataset505**. Follow-up DWI/ADC and outcome variables remain excluded.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json, os, shutil, subprocess

ROOT = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024')
ARCHIVE = ROOT / 'raw' / 'train.7z'
CORE_SOURCE = ROOT / 'core_source'
NNUNET_RAW = ROOT / 'nnUNet_raw'
RESULTS = ROOT / 'results'
AUDIT = RESULTS / 'ncct_perfusion_audit.json'
PREP_SUMMARY = RESULTS / 'dataset505_prep_summary.json'
REPO_ZIP = Path('/content/HybridStrokeSeg-main.zip')
REPO = Path('/content/HybridStrokeSeg-Reproduction-main')

if not ARCHIVE.is_file():
    raise FileNotFoundError(f'Missing verified ISLES24 archive: {ARCHIVE}')
CORE_SOURCE.mkdir(parents=True, exist_ok=True)
NNUNET_RAW.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

def run(cmd, *, cwd=None):
    cmd = [str(x) for x in cmd]
    print('$', ' '.join(cmd), flush=True)
    subprocess.run(cmd, check=True, cwd=cwd)


## Load latest repository and 7-Zip


In [ ]:
if REPO.exists():
    shutil.rmtree(REPO)
if REPO_ZIP.exists():
    REPO_ZIP.unlink()

run([
    'curl', '-fL', '--retry', '20', '--retry-delay', '3', '--retry-all-errors',
    'https://github.com/NTinkicht/HybridStrokeSeg-Reproduction/archive/refs/heads/main.zip',
    '-o', REPO_ZIP,
])
run(['unzip', '-q', REPO_ZIP, '-d', '/content'])
run(['python', '-m', 'pip', 'install', '-q', '-e', '.'], cwd=REPO)
run(['apt-get', 'update', '-qq'])
run(['apt-get', 'install', '-y', '-qq', 'p7zip-full'])


## Extract reusable acute core subset

Extraction is resumable by subject batches. It selects acute NCCT, registered CTA, registered perfusion maps, final labels and baseline clinical CSVs, while excluding raw 4-D CTP, follow-up MRI and outcomes.


In [ ]:
run([
    'python', 'scripts/extract_isles2024_core.py',
    ARCHIVE, CORE_SOURCE,
    '--batch-size', '10',
], cwd=REPO)


## Audit the exact Dataset505 imaging inputs


In [ ]:
run([
    'python', 'scripts/audit_isles2024.py', CORE_SOURCE,
    '--channels', 'ncct', 'tmax', 'cbf', 'cbv', 'mtt',
    '--json', AUDIT,
], cwd=REPO)

audit = json.loads(AUDIT.read_text())
required_checks = [
    audit['case_count_matches_release'],
    audit['ncct_present'] == 149,
    audit['tmax_present'] == 149 and audit['tmax_matches_ncct'] == 149,
    audit['cbf_present'] == 149 and audit['cbf_matches_ncct'] == 149,
    audit['cbv_present'] == 149 and audit['cbv_matches_ncct'] == 149,
    audit['mtt_present'] == 149 and audit['mtt_matches_ncct'] == 149,
    audit['lesion_mask_matches_ncct'] == 149,
]
if not all(required_checks):
    raise RuntimeError('Dataset505 geometry/completeness gate failed; staging is blocked.')
print('Audit gate passed for 149/149 cases and all five input channels.')


## Stage Dataset505 and enforce identical folds


In [ ]:
run([
    'python', 'scripts/stage_isles2024_nnunet.py', CORE_SOURCE, NNUNET_RAW,
    '--dataset-id', '505',
    '--dataset-name', 'ISLES2024_NCCT_PERFUSION',
    '--channels', 'ncct', 'tmax', 'cbf', 'cbv', 'mtt',
    '--mode', 'copy',
    '--resume',
    '--split-seed', '2026',
], cwd=REPO)

dataset504 = NNUNET_RAW / 'Dataset504_ISLES2024_NCCT'
dataset505 = NNUNET_RAW / 'Dataset505_ISLES2024_NCCT_PERFUSION'
run([
    'python', 'scripts/verify_nnunet_splits.py',
    dataset504 / 'splits_final.json',
    dataset505 / 'splits_final.json',
], cwd=REPO)


## Freeze preparation summary


In [ ]:
dataset_json = json.loads((dataset505 / 'dataset.json').read_text())
summary = {
    'dataset': dataset505.name,
    'num_training': dataset_json['numTraining'],
    'channel_names': dataset_json['channel_names'],
    'folds_identical_to_dataset504': True,
    'split_seed': 2026,
    'audit_path': str(AUDIT),
    'anti_leakage': (
        'Only acute NCCT + Tmax + CBF + CBV + MTT are model inputs; '
        'follow-up MRI and outcomes are excluded.'
    ),
}
PREP_SUMMARY.write_text(json.dumps(summary, indent=2) + '\n')
print(json.dumps(summary, indent=2))
print('\nDataset505 preparation complete:', dataset505)
